# my3d on Google Colab (free GPU)

Image in → 3D mesh (GLB) out.

**First:** *Runtime → Change runtime type → T4 GPU*, then run the cells top to bottom.
The first run takes about 10 minutes (install + model download).

In [1]:
# 1. Check the GPU (should show a Tesla T4 or better)
!nvidia-smi

Fri Nov 27 13:43:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Get the code
%cd /content
!git clone -q https://github.com/raj-anand-14/my3d
%cd /content/my3d
!git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG

/content
/content/my3d


In [3]:
# 3. Install (Colab already has PyTorch + CUDA; diso compiles GPU code, ~3-5 min)
!grep -v -E '^(diso|numpy)' TripoSG/requirements.txt > /tmp/req.txt
!pip install -q -r /tmp/req.txt -r requirements-app.txt
!pip install -q --no-build-isolation diso
print('Install done')

  Preparing metadata (pyproject.toml) ... done
Install done


In [ ]:
# 4. Start the my3d server in the background and wait for the models to load
import subprocess, time
log = open('/content/my3d/server.log', 'w')
proc = subprocess.Popen(['python', 'server.py'], cwd='/content/my3d', stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    time.sleep(5)
    txt = open('/content/my3d/server.log').read()
    if 'Uvicorn running' in txt or 'Application startup complete' in txt:
        print('Server is up!'); break
    if proc.poll() is not None:
        print(txt[-3000:]); raise RuntimeError('Server crashed, see log above')
else:
    print(open('/content/my3d/server.log').read()[-3000:])

In [ ]:
# 5. Open the web app (opens in a new tab: upload an image, get a 3D model)
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8000)

### Optional: generate without the web page
Upload an image, get a `.glb` downloaded straight to your computer.

In [ ]:
import requests
from google.colab import files
up = files.upload()
name = next(iter(up))
r = requests.post('http://localhost:8000/generate', files={'image': (name, up[name])}, data={'faces': 20000}, timeout=600)
r.raise_for_status()
out = name.rsplit('.', 1)[0] + '.glb'
open(out, 'wb').write(r.content)
files.download(out)

**If something fails:** run `!tail -50 /content/my3d/server.log` and share the output.